# Notebook 2: LightGBM training, ablations, final evaluation, exports

Project 4, retail demand forecasting on FreshRetailNet-50K. Continues notebook 1 but runs on its own: it downloads the
same pinned dataset and writes the same feature and evaluation modules.

This notebook:
1. Re-checks the data and confirms the baselines reproduce notebook 1's validation numbers exactly.
2. Builds features once with the serving code path and confirms masking realized data changes nothing.
3. Compares three LightGBM objectives on the validation week (squared error, absolute error, Tweedie).
4. Runs two ablations on the best objective: without same-day discount, and with oracle (actual) weather.
5. Retrains the served configuration on all 90 training days and scores the official eval week once.
6. Breaks errors down by horizon day, stockout hours, promotion flag, category, store and product.
7. Verifies serving parity end to end: predictions from a 34-day history snapshot equal the evaluation predictions.
8. Exports the model files, model card, metrics, parity sample and the serving history snapshot.

Runtime: CPU only. Kaggle setting required: Internet on.

## 1. Configuration

In [1]:
import hashlib
import json
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow as pa

DATASET_REPO = "Dingdong-Inc/FreshRetailNet-50K"
DATASET_REVISION = "08c1fab7f9257bc73679d415d65d644165d351d4"
DATASET_FILES = {
    "train": ("data/train.parquet", "6706832db892bbae4969c19d87e07975d2543d2ba7d7d4756360654785de5a3d"),
    "eval": ("data/eval.parquet", "1b118840664280c6b88bffc84c80ee1f54c05d911e354b7599e5da10995e960e"),
}
CACHE_DIR = Path.home() / ".cache" / "freshretailnet-50k" / DATASET_REVISION
KAGGLE_WORKING = Path("/kaggle/working")
OUTPUT_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODULE_DIR = Path.cwd()
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

EXPECTED_TRAIN_ROWS = 4_500_000
EXPECTED_EVAL_ROWS = 350_000
EXPECTED_SERIES = 50_000
HORIZON_DAYS = 7
SEED = 20240626
MAX_BOOST_ROUNDS = 3000
EARLY_STOPPING_ROUNDS = 100
NUM_THREADS = 4
SERVE_SAME_DAY_DISCOUNT = False
PARITY_SAMPLE_ROWS = 2000

NB1_VALIDATION_BASELINES = {
    "seasonal_naive_7": {"wape": 0.4123924065855388, "wpe": -0.11586909447557808, "mae": 0.49337008, "rmse": 0.8283414784702537},
    "moving_average_7": {"wape": 0.36563863424327187, "wpe": -0.11586909447557808, "mae": 0.43743570285714284, "rmse": 0.7387430908984557},
    "same_dow_mean_4w": {"wape": 0.39023538779530315, "wpe": -0.1232945722498342, "mae": 0.4668622928571428, "rmse": 0.7986253394610553},
}

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pa.__version__,
    "lightgbm": lgb.__version__,
    "on_kaggle": KAGGLE_WORKING.exists(),
}
print(json.dumps(environment, indent=2))
print("output dir:", OUTPUT_DIR)

report = {"checks": {}}


def check(name, condition, detail=None):
    report["checks"][name] = {"passed": bool(condition), "detail": detail}
    print(("PASS " if condition else "FAIL ") + name + ("" if detail is None else f" | {detail}"))
    assert condition, name

{
  "python": "3.11.15",
  "platform": "Linux-6.18.44-fc-v33-x86_64-with-glibc2.39",
  "numpy": "2.4.4",
  "pandas": "3.0.2",
  "pyarrow": "25.0.1",
  "lightgbm": "4.7.0",
  "on_kaggle": false
}
output dir: /home/claude/p4/run_nb2/outputs


## 2. Download at the pinned revision

In [2]:
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def fetch(split):
    rel, expected = DATASET_FILES[split]
    dest = CACHE_DIR / rel
    if not (dest.exists() and sha256_of(dest) == expected):
        dest.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://huggingface.co/datasets/{DATASET_REPO}/resolve/{DATASET_REVISION}/{rel}"
        tmp = dest.with_suffix(".part")
        with urllib.request.urlopen(url, timeout=120) as resp, open(tmp, "wb") as out:
            shutil.copyfileobj(resp, out, 1 << 20)
        tmp.replace(dest)
    actual = sha256_of(dest)
    assert actual == expected, f"{split}: sha256 mismatch {actual}"
    print(f"{split}: {dest.stat().st_size:,} bytes, sha256 ok")
    return dest


TRAIN_PATH = fetch("train")
EVAL_PATH = fetch("eval")

train: 106,436,287 bytes, sha256 ok
eval: 8,440,124 bytes, sha256 ok


## 3. Feature, evaluation and model modules

In [3]:
%%writefile demand_features.py
"""Leakage-safe daily demand features for FreshRetailNet-50K.

Every history-derived feature for target day t reads only days <= t - MIN_LAG.
With MIN_LAG equal to the forecast horizon, one feature definition serves every
horizon step 1..HORIZON from a single forecast origin (direct strategy), and the
same code path is used for training rows and for serving requests.

Same-day columns are split into two groups:
- KNOWN_FUTURE_COLS: planned in advance (calendar, discount, promo activity).
  Treated as known for the target day. Discount is an assumption, see notebook 1.
- Realized same-day columns (target, stockout hours, weather): never used for
  the target day. Weather is only available through the explicit "oracle" mode,
  which exists for a labelled sensitivity comparison and not for serving.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

KEY_COLS = ["store_id", "product_id"]
DATE_COL = "dt"
TARGET_COL = "sale_amount"
STOCK_COL = "stock_hour6_22_cnt"
STATIC_COLS = [
    "city_id",
    "store_id",
    "management_group_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
    "product_id",
]
KNOWN_FUTURE_COLS = ["discount", "activity_flag", "holiday_flag"]
WEATHER_COLS = ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
REALIZED_COLS = [TARGET_COL, STOCK_COL] + WEATHER_COLS

HORIZON = 7
MIN_LAG = HORIZON
SALES_LAGS = (7, 8, 9, 10, 11, 12, 13, 14, 21, 28)
SALES_MEAN_WINDOWS = (7, 14, 28)
SALES_STD_WINDOWS = (7, 28)
STOCK_MEAN_WINDOWS = (7, 28)
FULL_DAY_STOCKOUT_HOURS = 16
WEATHER_MODES = ("none", "oracle")

REQUIRED_HISTORY_DAYS = max(max(SALES_LAGS), MIN_LAG + max(SALES_MEAN_WINDOWS) - 1)


class PanelError(ValueError):
    pass


def to_panel(df: pd.DataFrame, value_cols: list[str]) -> tuple[pd.DataFrame, pd.DatetimeIndex, dict[str, np.ndarray]]:
    """Reshape a long frame into (n_series, n_days) arrays.

    Requires a complete, duplicate-free panel over a contiguous daily range.
    """
    missing = [c for c in KEY_COLS + [DATE_COL] + value_cols if c not in df.columns]
    if missing:
        raise PanelError(f"missing columns: {missing}")
    dates = pd.to_datetime(df[DATE_COL])
    start, end = dates.min(), dates.max()
    grid = pd.date_range(start, end, freq="D")
    n_days = len(grid)
    keys = df[KEY_COLS].drop_duplicates().sort_values(KEY_COLS).reset_index(drop=True)
    n_series = len(keys)
    if len(df) != n_series * n_days:
        raise PanelError(
            f"incomplete or duplicated panel: {len(df)} rows, expected {n_series} x {n_days} = {n_series * n_days}"
        )
    order = np.lexsort((dates.to_numpy(), df["product_id"].to_numpy(), df["store_id"].to_numpy()))
    sorted_dates = dates.to_numpy()[order].reshape(n_series, n_days)
    if not (sorted_dates == grid.to_numpy()[None, :]).all():
        raise PanelError("panel dates are not a complete contiguous grid for every series")
    sorted_keys = df[KEY_COLS].to_numpy()[order].reshape(n_series, n_days, len(KEY_COLS))
    if not (sorted_keys == sorted_keys[:, :1, :]).all():
        raise PanelError("series keys are not constant along the date axis after sorting")
    arrays = {c: df[c].to_numpy(dtype=np.float64)[order].reshape(n_series, n_days) for c in value_cols}
    return keys, grid, arrays


def lag(x: np.ndarray, k: int) -> np.ndarray:
    out = np.full_like(x, np.nan, dtype=np.float64)
    if k < x.shape[1]:
        out[:, k:] = x[:, : x.shape[1] - k]
    return out


def _window_sums(x: np.ndarray, window: int, offset: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Sums over days (t - offset - window + 1 .. t - offset), accumulated window element by element.

    Each output value depends only on the values inside its own window, so the result is bit-identical
    regardless of how much history precedes the window (required for serving parity).
    """
    s1 = np.zeros(x.shape, dtype=np.float64)
    s2 = np.zeros(x.shape, dtype=np.float64)
    n = np.zeros(x.shape, dtype=np.int32)
    for j in range(window):
        v = lag(x, offset + j)
        valid = ~np.isnan(v)
        v0 = np.where(valid, v, 0.0)
        s1 += v0
        s2 += v0 * v0
        n += valid
    return s1, s2, n


def rolling_mean(x: np.ndarray, window: int, offset: int) -> np.ndarray:
    """Mean over days (t - offset - window + 1 .. t - offset). NaN unless the full window is observed."""
    s1, _, n = _window_sums(x, window, offset)
    return np.where(n == window, s1 / window, np.nan)


def rolling_std(x: np.ndarray, window: int, offset: int) -> np.ndarray:
    """Population standard deviation over the same window as rolling_mean."""
    s1, s2, n = _window_sums(x, window, offset)
    mean = s1 / window
    var = np.maximum(s2 / window - mean * mean, 0.0)
    return np.where(n == window, np.sqrt(var), np.nan)


def feature_columns(weather_mode: str = "none") -> list[str]:
    if weather_mode not in WEATHER_MODES:
        raise ValueError(f"weather_mode must be one of {WEATHER_MODES}")
    cols = list(STATIC_COLS) + ["day_of_week"] + list(KNOWN_FUTURE_COLS)
    cols += [f"sales_lag_{k}" for k in SALES_LAGS]
    cols += [f"sales_mean_{w}_off{MIN_LAG}" for w in SALES_MEAN_WINDOWS]
    cols += [f"sales_std_{w}_off{MIN_LAG}" for w in SALES_STD_WINDOWS]
    cols += ["sales_same_dow_mean_4w", f"sales_zero_share_28_off{MIN_LAG}"]
    cols += [f"stock_hours_lag_{MIN_LAG}"]
    cols += [f"stock_hours_mean_{w}_off{MIN_LAG}" for w in STOCK_MEAN_WINDOWS]
    cols += [f"stock_fullday_share_28_off{MIN_LAG}"]
    cols += [f"discount_mean_7_off{MIN_LAG}", "discount_vs_recent", f"activity_share_28_off{MIN_LAG}"]
    if weather_mode == "oracle":
        cols += [f"oracle_{c}" for c in WEATHER_COLS]
    return cols


def build_feature_arrays(
    arrays: dict[str, np.ndarray],
    grid: pd.DatetimeIndex,
    weather_mode: str = "none",
) -> dict[str, np.ndarray]:
    """History and known-future features as (n_series, n_days) arrays. Static columns excluded."""
    if weather_mode not in WEATHER_MODES:
        raise ValueError(f"weather_mode must be one of {WEATHER_MODES}")
    y = arrays[TARGET_COL]
    stock = arrays[STOCK_COL]
    n_series, n_days = y.shape
    f: dict[str, np.ndarray] = {}

    f["day_of_week"] = np.broadcast_to(grid.dayofweek.to_numpy(dtype=np.float64)[None, :], (n_series, n_days))
    for c in KNOWN_FUTURE_COLS:
        f[c] = arrays[c]

    for k in SALES_LAGS:
        f[f"sales_lag_{k}"] = lag(y, k)
    for w in SALES_MEAN_WINDOWS:
        f[f"sales_mean_{w}_off{MIN_LAG}"] = rolling_mean(y, w, MIN_LAG)
    for w in SALES_STD_WINDOWS:
        f[f"sales_std_{w}_off{MIN_LAG}"] = rolling_std(y, w, MIN_LAG)
    f["sales_same_dow_mean_4w"] = (lag(y, 7) + lag(y, 14) + lag(y, 21) + lag(y, 28)) / 4.0
    zero = np.where(np.isnan(y), np.nan, (y == 0).astype(np.float64))
    f[f"sales_zero_share_28_off{MIN_LAG}"] = rolling_mean(zero, 28, MIN_LAG)

    f[f"stock_hours_lag_{MIN_LAG}"] = lag(stock, MIN_LAG)
    for w in STOCK_MEAN_WINDOWS:
        f[f"stock_hours_mean_{w}_off{MIN_LAG}"] = rolling_mean(stock, w, MIN_LAG)
    fullday = np.where(np.isnan(stock), np.nan, (stock >= FULL_DAY_STOCKOUT_HOURS).astype(np.float64))
    f[f"stock_fullday_share_28_off{MIN_LAG}"] = rolling_mean(fullday, 28, MIN_LAG)

    disc_recent = rolling_mean(arrays["discount"], 7, MIN_LAG)
    f[f"discount_mean_7_off{MIN_LAG}"] = disc_recent
    f["discount_vs_recent"] = arrays["discount"] - disc_recent
    f[f"activity_share_28_off{MIN_LAG}"] = rolling_mean(arrays["activity_flag"], 28, MIN_LAG)

    if weather_mode == "oracle":
        for c in WEATHER_COLS:
            f[f"oracle_{c}"] = arrays[c]
    return f


def mask_realized(arrays: dict[str, np.ndarray], grid: pd.DatetimeIndex, first_masked_date) -> dict[str, np.ndarray]:
    """Copy of arrays with realized columns set to NaN from first_masked_date onward."""
    idx = int(np.searchsorted(grid.to_numpy(), np.datetime64(pd.Timestamp(first_masked_date))))
    out = dict(arrays)
    for c in REALIZED_COLS:
        if c in out:
            a = out[c].copy()
            a[:, idx:] = np.nan
            out[c] = a
    return out


def panel_value_cols(weather_mode: str = "none") -> list[str]:
    cols = [TARGET_COL, STOCK_COL] + list(KNOWN_FUTURE_COLS)
    if weather_mode == "oracle":
        cols += list(WEATHER_COLS)
    return cols


def build_features(
    df: pd.DataFrame,
    weather_mode: str = "none",
    first_masked_date=None,
    rows_from_date=None,
) -> pd.DataFrame:
    """Long feature frame: keys, dt, feature_columns(weather_mode).

    first_masked_date: realized columns from this date on are removed before any
    feature is computed. rows_from_date: only rows on or after this date are returned.
    """
    keys, grid, arrays = to_panel(df, panel_value_cols(weather_mode))
    if first_masked_date is not None:
        arrays = mask_realized(arrays, grid, first_masked_date)
    feats = build_feature_arrays(arrays, grid, weather_mode)

    static = df.drop_duplicates(KEY_COLS)[STATIC_COLS].copy()
    static = keys.merge(static, on=KEY_COLS, how="left", validate="one_to_one")
    if static[STATIC_COLS].isna().any().any():
        raise PanelError("static attributes missing for some series")

    start = 0
    if rows_from_date is not None:
        start = int(np.searchsorted(grid.to_numpy(), np.datetime64(pd.Timestamp(rows_from_date))))
    n_series = len(keys)
    days = grid[start:]
    n_days = len(days)

    out = {}
    for c in STATIC_COLS:
        out[c] = np.repeat(static[c].to_numpy(), n_days)
    out[DATE_COL] = np.tile(days.to_numpy(), n_series)
    for name in feature_columns(weather_mode):
        if name in STATIC_COLS:
            continue
        out[name] = np.ascontiguousarray(feats[name][:, start:]).reshape(-1).astype(np.float32)
    frame = pd.DataFrame(out)
    return frame[STATIC_COLS + [DATE_COL] + [c for c in feature_columns(weather_mode) if c not in STATIC_COLS]]

Writing demand_features.py


In [4]:
%%writefile demand_eval.py
"""Baselines and metrics for fixed-origin 7-day forecasts.

All baselines receive the target panel with every day after the forecast origin
set to NaN, and fail if any prediction is non-finite, so a baseline that reads
past the origin cannot silently produce numbers.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from demand_features import HORIZON

BASELINES = ("seasonal_naive_7", "moving_average_7", "same_dow_mean_4w")


def _masked(y: np.ndarray, origin_idx: int) -> np.ndarray:
    m = y.astype(np.float64, copy=True)
    m[:, origin_idx + 1 :] = np.nan
    return m


def baseline_forecast(y: np.ndarray, origin_idx: int, name: str, horizon: int = HORIZON) -> np.ndarray:
    """Forecast of shape (n_series, horizon) for days origin_idx+1 .. origin_idx+horizon."""
    if origin_idx + 1 < 28:
        raise ValueError("origin needs at least 28 observed days")
    hist = _masked(y, origin_idx)
    targets = origin_idx + 1 + np.arange(horizon)
    if name == "seasonal_naive_7":
        pred = hist[:, targets - 7]
    elif name == "moving_average_7":
        pred = np.repeat(hist[:, origin_idx - 6 : origin_idx + 1].mean(axis=1, keepdims=True), horizon, axis=1)
    elif name == "same_dow_mean_4w":
        pred = np.mean(np.stack([hist[:, targets - k] for k in (7, 14, 21, 28)]), axis=0)
    else:
        raise ValueError(f"unknown baseline {name}")
    if not np.isfinite(pred).all():
        raise RuntimeError(f"{name} produced non-finite values; it read data after the origin")
    return pred


def point_metrics(actual, pred) -> dict[str, float]:
    a = np.asarray(actual, dtype=np.float64)
    p = np.asarray(pred, dtype=np.float64)
    err = p - a
    total = a.sum()
    return {
        "n": int(a.size),
        "sum_actual": float(total),
        "wape": float(np.abs(err).sum() / total) if total > 0 else float("nan"),
        "wpe": float(err.sum() / total) if total > 0 else float("nan"),
        "mae": float(np.abs(err).mean()),
        "rmse": float(np.sqrt((err * err).mean())),
    }


def metrics_by(frame: pd.DataFrame, by: str, actual: str = "actual", pred: str = "pred") -> pd.DataFrame:
    rows = []
    for key, g in frame.groupby(by, observed=True, sort=True):
        rows.append({by: key, **point_metrics(g[actual].to_numpy(), g[pred].to_numpy())})
    return pd.DataFrame(rows)


def stockout_bucket(hours) -> pd.Categorical:
    h = np.asarray(hours)
    labels = np.where(h == 0, "0h", np.where(h >= 16, "16h_full_day", "1-15h"))
    return pd.Categorical(labels, categories=["0h", "1-15h", "16h_full_day"], ordered=True)

Writing demand_eval.py


In [5]:
%%writefile demand_model.py
"""LightGBM training and prediction contract shared by notebook 2 and the serving layer.

The model consumes a float32 matrix whose columns follow model_feature_columns() exactly.
Predictions are clipped at zero because sales are non-negative.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from demand_features import feature_columns

CATEGORICAL_FEATURES = [
    "city_id",
    "store_id",
    "management_group_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
    "product_id",
    "day_of_week",
]
SAME_DAY_DISCOUNT_FEATURES = ["discount", "discount_vs_recent"]

BASE_PARAMS = {
    "metric": "l1",
    "learning_rate": 0.05,
    "num_leaves": 255,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "max_bin": 255,
    "seed": 20240626,
    "deterministic": True,
    "force_row_wise": True,
    "num_threads": 4,
    "verbose": -1,
}
OBJECTIVES = {
    "l2": {"objective": "regression"},
    "l1": {"objective": "regression_l1"},
    "tweedie": {"objective": "tweedie", "tweedie_variance_power": 1.2},
}


def model_feature_columns(weather_mode: str = "none", same_day_discount: bool = True) -> list[str]:
    cols = feature_columns(weather_mode)
    if not same_day_discount:
        cols = [c for c in cols if c not in SAME_DAY_DISCOUNT_FEATURES]
    return cols


def model_params(objective: str) -> dict:
    if objective not in OBJECTIVES:
        raise ValueError(f"objective must be one of {sorted(OBJECTIVES)}")
    return {**BASE_PARAMS, **OBJECTIVES[objective]}


def to_matrix(frame: pd.DataFrame, columns: list[str]) -> np.ndarray:
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(f"feature columns missing: {missing}")
    return frame[columns].to_numpy(dtype=np.float32)


def make_dataset(x: np.ndarray, y: np.ndarray, columns: list[str], reference=None):
    import lightgbm as lgb

    if x.shape[1] != len(columns):
        raise ValueError("matrix width does not match column list")
    return lgb.Dataset(
        x,
        label=y,
        feature_name=list(columns),
        categorical_feature=[c for c in CATEGORICAL_FEATURES if c in columns],
        reference=reference,
        free_raw_data=True,
    )


def train(
    x_train: np.ndarray,
    y_train: np.ndarray,
    columns: list[str],
    objective: str,
    num_boost_round: int,
    x_valid: np.ndarray | None = None,
    y_valid: np.ndarray | None = None,
    early_stopping_rounds: int | None = None,
    num_threads: int | None = None,
):
    import lightgbm as lgb

    params = model_params(objective)
    if num_threads is not None:
        params["num_threads"] = num_threads
    dtrain = make_dataset(x_train, y_train, columns)
    callbacks = []
    valid_sets = []
    if x_valid is not None:
        valid_sets = [make_dataset(x_valid, y_valid, columns, reference=dtrain)]
        if early_stopping_rounds:
            callbacks.append(lgb.early_stopping(early_stopping_rounds, first_metric_only=True, verbose=False))
    booster = lgb.train(params, dtrain, num_boost_round=num_boost_round, valid_sets=valid_sets, callbacks=callbacks)
    return booster


def predict(booster, frame_or_matrix, columns: list[str] | None = None, num_iteration: int | None = None) -> np.ndarray:
    names = booster.feature_name()
    if isinstance(frame_or_matrix, pd.DataFrame):
        x = to_matrix(frame_or_matrix, names)
    else:
        if columns is None or list(columns) != names:
            raise ValueError("matrix input requires columns equal to the booster feature order")
        x = np.asarray(frame_or_matrix, dtype=np.float32)
    if num_iteration is None:
        best = booster.best_iteration
        num_iteration = best if best and best > 0 else None
    return np.clip(booster.predict(x, num_iteration=num_iteration), 0.0, None)

Writing demand_model.py


In [6]:
%%writefile test_demand_features.py
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path(__file__).resolve().parent
for _candidate in (_here, _here.parent / "src"):
    if (_candidate / "demand_features.py").exists():
        sys.path.insert(0, str(_candidate))
        break

import demand_eval as de
import demand_features as dfe


def synthetic_long(n_stores=3, n_products=4, n_days=70, seed=0):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2024-01-01", periods=n_days, freq="D")
    rows = []
    for s in range(n_stores):
        for p in range(n_products):
            rows.append(
                pd.DataFrame(
                    {
                        "city_id": s % 2,
                        "store_id": s,
                        "management_group_id": p % 3,
                        "first_category_id": p % 2,
                        "second_category_id": p % 3,
                        "third_category_id": p,
                        "product_id": p,
                        "dt": dates.strftime("%Y-%m-%d"),
                        "sale_amount": rng.gamma(2.0, 1.0, n_days).round(1),
                        "stock_hour6_22_cnt": rng.integers(0, 17, n_days),
                        "discount": rng.uniform(0.5, 1.0, n_days).round(3),
                        "activity_flag": rng.integers(0, 2, n_days),
                        "holiday_flag": (dates.dayofweek >= 5).astype(int),
                        "precpt": rng.uniform(0, 10, n_days),
                        "avg_temperature": rng.uniform(15, 30, n_days),
                        "avg_humidity": rng.uniform(40, 90, n_days),
                        "avg_wind_level": rng.uniform(0, 4, n_days),
                    }
                )
            )
    df = pd.concat(rows, ignore_index=True)
    first = (df["store_id"] == 0) & (df["product_id"] == 0)
    day = np.arange(first.sum())
    df.loc[first, "sale_amount"] = np.where(day >= len(day) - 50, 0.1, np.round(rng.gamma(5.0, 7.3, first.sum()), 1))
    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def _features(df, mode="none"):
    keys, grid, arrays = dfe.to_panel(df, dfe.panel_value_cols(mode))
    return keys, grid, arrays, dfe.build_feature_arrays(arrays, grid, mode)


def test_lag_and_window_values():
    df = synthetic_long()
    keys, grid, arrays, f = _features(df)
    y = arrays["sale_amount"]
    t = 40
    assert np.allclose(f["sales_lag_7"][:, t], y[:, t - 7])
    assert np.allclose(f["sales_lag_28"][:, t], y[:, t - 28])
    assert np.allclose(f["sales_mean_7_off7"][:, t], y[:, t - 13 : t - 6].mean(axis=1))
    assert np.allclose(f["sales_mean_28_off7"][:, t], y[:, t - 34 : t - 6].mean(axis=1))
    assert np.allclose(f["sales_std_7_off7"][:, t], y[:, t - 13 : t - 6].std(axis=1), atol=1e-6)
    assert np.isnan(f["sales_mean_28_off7"][:, 33]).all()
    assert np.isfinite(f["sales_mean_28_off7"][:, 34]).all()


def test_target_perturbation_does_not_reach_next_six_days():
    df = synthetic_long()
    keys, grid, arrays, base = _features(df)
    d = 45
    pert = dict(arrays)
    pert["sale_amount"] = arrays["sale_amount"].copy()
    pert["sale_amount"][:, d] += 100.0
    f = dfe.build_feature_arrays(pert, grid, "none")
    for name in base:
        assert np.array_equal(base[name][:, d : d + 7], f[name][:, d : d + 7], equal_nan=True), name
    changed = [n for n in base if not np.array_equal(base[n][:, d + 7], f[n][:, d + 7], equal_nan=True)]
    assert "sales_lag_7" in changed and "sales_mean_7_off7" in changed


def test_same_day_realized_columns_are_not_features():
    df = synthetic_long()
    keys, grid, arrays, base = _features(df, "none")
    d = 50
    pert = {k: v.copy() for k, v in arrays.items()}
    pert["stock_hour6_22_cnt"][:, d:] = 16.0
    pert["sale_amount"][:, d:] = 999.0
    f = dfe.build_feature_arrays(pert, grid, "none")
    for name in base:
        assert np.array_equal(base[name][:, d : d + 7], f[name][:, d : d + 7], equal_nan=True), name


def test_oracle_weather_is_the_only_mode_reading_same_day_weather():
    df = synthetic_long()
    keys, grid, arrays, base = _features(df, "oracle")
    pert = {k: v.copy() for k, v in arrays.items()}
    pert["precpt"][:, 50] += 5.0
    f = dfe.build_feature_arrays(pert, grid, "oracle")
    assert not np.array_equal(base["oracle_precpt"][:, 50], f["oracle_precpt"][:, 50])
    assert not any(c.startswith("oracle_") for c in dfe.feature_columns("none"))


def test_masking_matches_unmasked_for_horizon_rows():
    df = synthetic_long()
    first_masked = pd.Timestamp("2024-01-01") + pd.Timedelta(days=63)
    full = dfe.build_features(df, rows_from_date=first_masked)
    masked = dfe.build_features(df, first_masked_date=first_masked, rows_from_date=first_masked)
    pd.testing.assert_frame_equal(full, masked, check_exact=True)


def test_truncated_history_gives_identical_horizon_features():
    df = synthetic_long(n_days=200)
    horizon_start = pd.Timestamp("2024-01-01") + pd.Timedelta(days=193)
    full = dfe.build_features(df, rows_from_date=horizon_start)
    keep_from = horizon_start - pd.Timedelta(days=dfe.REQUIRED_HISTORY_DAYS)
    short = df[pd.to_datetime(df["dt"]) >= keep_from]
    trunc = dfe.build_features(short, rows_from_date=horizon_start)
    pd.testing.assert_frame_equal(full, trunc, check_exact=True)
    too_short = df[pd.to_datetime(df["dt"]) >= keep_from + pd.Timedelta(days=1)]
    trunc2 = dfe.build_features(too_short, rows_from_date=horizon_start)
    assert trunc2.isna().sum().sum() > full.isna().sum().sum()


def test_incomplete_panel_raises():
    df = synthetic_long()
    try:
        dfe.to_panel(df.iloc[1:], dfe.panel_value_cols())
    except dfe.PanelError:
        return
    raise AssertionError("expected PanelError")


def test_baselines_ignore_data_after_origin():
    df = synthetic_long()
    keys, grid, arrays = dfe.to_panel(df, dfe.panel_value_cols())
    y = arrays["sale_amount"]
    origin = 55
    pert = y.copy()
    pert[:, origin + 1 :] = -1.0e6
    for name in de.BASELINES:
        a = de.baseline_forecast(y, origin, name)
        b = de.baseline_forecast(pert, origin, name)
        assert np.array_equal(a, b), name
    assert np.array_equal(de.baseline_forecast(y, origin, "seasonal_naive_7")[:, 0], y[:, origin - 6])


def test_metrics_values():
    m = de.point_metrics([1.0, 3.0], [2.0, 1.0])
    assert np.isclose(m["wape"], 3.0 / 4.0)
    assert np.isclose(m["wpe"], -1.0 / 4.0)
    assert np.isclose(m["mae"], 1.5)
    assert np.isclose(m["rmse"], np.sqrt(2.5))


if __name__ == "__main__":
    names = [n for n in sorted(globals()) if n.startswith("test_")]
    for n in names:
        globals()[n]()
        print(f"PASS {n}")
    print(f"{len(names)} passed")

Writing test_demand_features.py


In [7]:
%%writefile test_demand_model.py
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path(__file__).resolve().parent
for _candidate in (_here, _here.parent / "src"):
    if (_candidate / "demand_model.py").exists():
        sys.path.insert(0, str(_candidate))
        break

import demand_features as dfe
import demand_model as dm


def _synthetic(n_rows=4000, seed=0):
    rng = np.random.default_rng(seed)
    cols = dm.model_feature_columns("none", True)
    frame = pd.DataFrame({c: rng.uniform(0, 5, n_rows) for c in cols})
    for c in dm.CATEGORICAL_FEATURES:
        frame[c] = rng.integers(0, 6, n_rows)
    y = np.maximum(frame["sales_lag_7"].to_numpy() * 0.8 + rng.normal(0, 0.3, n_rows), 0)
    return frame, y, cols


def test_discount_variant_removes_only_same_day_discount():
    with_d = dm.model_feature_columns("none", True)
    without_d = dm.model_feature_columns("none", False)
    assert set(with_d) - set(without_d) == set(dm.SAME_DAY_DISCOUNT_FEATURES)
    assert "discount_mean_7_off7" in without_d
    assert not any(c.startswith("oracle_") for c in with_d + without_d)


def test_categorical_features_are_model_inputs():
    assert set(dm.CATEGORICAL_FEATURES) <= set(dfe.feature_columns("none"))


def test_train_predict_roundtrip_and_clipping(tmp_path=None):
    frame, y, cols = _synthetic()
    x = dm.to_matrix(frame, cols)
    booster = dm.train(x[:3000], y[:3000], cols, "l2", 50, x[3000:], y[3000:], early_stopping_rounds=10, num_threads=1)
    pred = dm.predict(booster, frame.iloc[3000:])
    assert pred.shape == (1000,) and (pred >= 0).all()
    shuffled = frame.iloc[3000:][list(reversed(cols))]
    assert np.array_equal(pred, dm.predict(booster, shuffled))
    text = booster.model_to_string(num_iteration=-1)
    import lightgbm as lgb

    reloaded = lgb.Booster(model_str=text)
    assert np.array_equal(pred, dm.predict(reloaded, frame.iloc[3000:], num_iteration=booster.best_iteration))


def test_matrix_predict_rejects_wrong_column_order():
    frame, y, cols = _synthetic(600)
    x = dm.to_matrix(frame, cols)
    booster = dm.train(x, y, cols, "l2", 5, num_threads=1)
    try:
        dm.predict(booster, x, columns=list(reversed(cols)))
    except ValueError:
        return
    raise AssertionError("expected ValueError for wrong column order")


def test_missing_feature_column_raises():
    frame, y, cols = _synthetic(600)
    x = dm.to_matrix(frame, cols)
    booster = dm.train(x, y, cols, "l2", 5, num_threads=1)
    try:
        dm.predict(booster, frame.drop(columns=["sales_lag_7"]))
    except KeyError:
        return
    raise AssertionError("expected KeyError for missing column")


if __name__ == "__main__":
    names = [n for n in sorted(globals()) if n.startswith("test_")]
    for n in names:
        globals()[n]()
        print(f"PASS {n}")
    print(f"{len(names)} passed")

Writing test_demand_model.py


In [8]:
for test_file in ("test_demand_features.py", "test_demand_model.py"):
    result = subprocess.run([sys.executable, test_file], capture_output=True, text=True, cwd=MODULE_DIR)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr[-3000:])
    check(f"{test_file} passes", result.returncode == 0)

import demand_eval as de
import demand_features as dfe
import demand_model as dm

PASS test_baselines_ignore_data_after_origin
PASS test_incomplete_panel_raises
PASS test_lag_and_window_values
PASS test_masking_matches_unmasked_for_horizon_rows
PASS test_metrics_values
PASS test_oracle_weather_is_the_only_mode_reading_same_day_weather
PASS test_same_day_realized_columns_are_not_features
PASS test_target_perturbation_does_not_reach_next_six_days
PASS test_truncated_history_gives_identical_horizon_features
9 passed

PASS test_demand_features.py passes


PASS test_categorical_features_are_model_inputs
PASS test_discount_variant_removes_only_same_day_discount
PASS test_matrix_predict_rejects_wrong_column_order
PASS test_missing_feature_column_raises
PASS test_train_predict_roundtrip_and_clipping
5 passed

PASS test_demand_model.py passes


## 4. Load, structural re-check, feature build

In [9]:
load_cols = list(dict.fromkeys(dfe.STATIC_COLS + [dfe.DATE_COL] + dfe.panel_value_cols("oracle")))
train = pd.read_parquet(TRAIN_PATH, columns=load_cols)
evald = pd.read_parquet(EVAL_PATH, columns=load_cols)
check("train row count", len(train) == EXPECTED_TRAIN_ROWS, len(train))
check("eval row count", len(evald) == EXPECTED_EVAL_ROWS, len(evald))
check("no nulls", int(train.isna().sum().sum()) == 0 and int(evald.isna().sum().sum()) == 0)

combined = pd.concat([train, evald], ignore_index=True)
del train, evald
keys, grid, panel = dfe.to_panel(combined, dfe.panel_value_cols("oracle"))
n_series, n_days = len(keys), len(grid)
check("combined panel is complete", n_series == EXPECTED_SERIES and n_days == 97, (n_series, n_days))

EVAL_START_IDX = n_days - HORIZON_DAYS
VALIDATION_START_IDX = EVAL_START_IDX - HORIZON_DAYS
FIRST_TRAIN_IDX = dfe.REQUIRED_HISTORY_DAYS
EVAL_START, VALIDATION_START = grid[EVAL_START_IDX], grid[VALIDATION_START_IDX]
split = {
    "fit_target_days": [str(grid[FIRST_TRAIN_IDX].date()), str(grid[VALIDATION_START_IDX - 1].date())],
    "validation_days": [str(VALIDATION_START.date()), str(grid[EVAL_START_IDX - 1].date())],
    "final_train_target_days": [str(grid[FIRST_TRAIN_IDX].date()), str(grid[EVAL_START_IDX - 1].date())],
    "eval_days": [str(EVAL_START.date()), str(grid[-1].date())],
    "first_train_day_reason": f"first day with a complete {dfe.REQUIRED_HISTORY_DAYS}-day feature history",
}
check("split dates", split["validation_days"] == ["2024-06-19", "2024-06-25"] and split["eval_days"] == ["2024-06-26", "2024-07-02"], split)
report["split"] = split

t0 = time.time()
features = dfe.build_features(combined, weather_mode="oracle")
print(f"feature build {time.time() - t0:.1f}s, shape {features.shape}")
check("feature rows are series-major and aligned with the panel",
      np.array_equal(features["store_id"].to_numpy(), np.repeat(keys["store_id"].to_numpy(), n_days))
      and np.array_equal(features["product_id"].to_numpy(), np.repeat(keys["product_id"].to_numpy(), n_days))
      and np.array_equal(features[dfe.DATE_COL].to_numpy(), np.tile(grid.to_numpy(), n_series)))

non_oracle = [c for c in dfe.feature_columns("none")]
for label, start in (("validation", VALIDATION_START), ("eval", EVAL_START)):
    end = start + pd.Timedelta(days=HORIZON_DAYS)
    masked = dfe.build_features(combined, weather_mode="none", first_masked_date=start, rows_from_date=start)
    masked = masked[masked[dfe.DATE_COL] < end].reset_index(drop=True)
    in_window = (features[dfe.DATE_COL] >= start) & (features[dfe.DATE_COL] < end)
    unmasked = features.loc[in_window, list(masked.columns)].reset_index(drop=True)
    pd.testing.assert_frame_equal(masked, unmasked, check_exact=True)
    check(f"{label} week features identical with realized data masked from {start.date()}", True, f"{len(masked):,} rows")
    del masked, unmasked

PASS train row count | 4500000
PASS eval row count | 350000
PASS no nulls


PASS combined panel is complete | (50000, 97)
PASS split dates | {'fit_target_days': ['2024-05-01', '2024-06-18'], 'validation_days': ['2024-06-19', '2024-06-25'], 'final_train_target_days': ['2024-05-01', '2024-06-25'], 'eval_days': ['2024-06-26', '2024-07-02'], 'first_train_day_reason': 'first day with a complete 34-day feature history'}


feature build 14.6s, shape (4850000, 40)
PASS feature rows are series-major and aligned with the panel


PASS validation week features identical with realized data masked from 2024-06-19 | 350,000 rows


PASS eval week features identical with realized data masked from 2024-06-26 | 350,000 rows


In [10]:
ALL_MODEL_COLUMNS = dm.model_feature_columns("oracle", True)
X_all = dm.to_matrix(features, ALL_MODEL_COLUMNS)
col_index = {c: i for i, c in enumerate(ALL_MODEL_COLUMNS)}
meta = pd.DataFrame({
    "store_id": features["store_id"].to_numpy(),
    "product_id": features["product_id"].to_numpy(),
    "first_category_id": features["first_category_id"].to_numpy(),
    "dt": features[dfe.DATE_COL].to_numpy(),
    "day_idx": np.tile(np.arange(n_days), n_series),
})
y_all = panel[dfe.TARGET_COL].reshape(-1)
stock_all = panel[dfe.STOCK_COL].reshape(-1)
activity_all = panel["activity_flag"].reshape(-1).astype(int)
del features

day_idx = meta["day_idx"].to_numpy()
rows_fit = np.flatnonzero((day_idx >= FIRST_TRAIN_IDX) & (day_idx < VALIDATION_START_IDX))
rows_val = np.flatnonzero((day_idx >= VALIDATION_START_IDX) & (day_idx < EVAL_START_IDX))
rows_final = np.flatnonzero((day_idx >= FIRST_TRAIN_IDX) & (day_idx < EVAL_START_IDX))
rows_eval = np.flatnonzero(day_idx >= EVAL_START_IDX)
none_cols = [col_index[c] for c in dm.model_feature_columns("none", True)]
check("no missing feature values on any training, validation or eval row",
      not np.isnan(X_all[np.concatenate([rows_final, rows_eval])][:, none_cols]).any())
check("row counts", (len(rows_fit), len(rows_val), len(rows_final), len(rows_eval)) ==
      (49 * n_series, 7 * n_series, 56 * n_series, 7 * n_series),
      {"fit": len(rows_fit), "validation": len(rows_val), "final_train": len(rows_final), "eval": len(rows_eval)})

PASS no missing feature values on any training, validation or eval row
PASS row counts | {'fit': 2450000, 'validation': 350000, 'final_train': 2800000, 'eval': 350000}


## 5. Baselines on validation, regression check against notebook 1

The baselines must reproduce notebook 1's validation metrics. A mismatch means the data or code differ between the two
notebooks.

In [11]:
def baseline_predictions(origin_idx):
    y_panel = panel[dfe.TARGET_COL]
    return {name: de.baseline_forecast(y_panel, origin_idx, name).reshape(-1) for name in de.BASELINES}


def frame_for(rows, pred):
    return pd.DataFrame({
        "store_id": meta["store_id"].to_numpy()[rows],
        "product_id": meta["product_id"].to_numpy()[rows],
        "first_category_id": meta["first_category_id"].to_numpy()[rows],
        "horizon_day": day_idx[rows] - (day_idx[rows].min() - 1),
        "stockout_bucket": de.stockout_bucket(stock_all[rows]),
        "activity_flag": activity_all[rows],
        "actual": y_all[rows],
        "pred": pred,
    })


def breakdowns(frame):
    out = {}
    for by in ("horizon_day", "stockout_bucket", "activity_flag", "first_category_id"):
        t = de.metrics_by(frame, by)
        t[by] = t[by].astype(str)
        out[f"by_{by}"] = t.to_dict(orient="records")
    return out


def worst_groups(frame, by, top=10):
    g = frame.assign(abs_err=(frame["pred"] - frame["actual"]).abs()).groupby(by).agg(
        rows=("actual", "size"), sum_actual=("actual", "sum"), sum_pred=("pred", "sum"), abs_error=("abs_err", "sum"))
    g["wape"] = g["abs_error"] / g["sum_actual"]
    g["wpe"] = (g["sum_pred"] - g["sum_actual"]) / g["sum_actual"]
    return g.sort_values("abs_error", ascending=False).head(top).reset_index()


val_baselines = baseline_predictions(VALIDATION_START_IDX - 1)
validation_results = {}
for name, pred in val_baselines.items():
    m = de.point_metrics(y_all[rows_val], pred)
    validation_results[name] = {"kind": "baseline", "overall": m}
    expected = NB1_VALIDATION_BASELINES[name]
    same = all(abs(m[k] - expected[k]) <= 1e-9 * max(1.0, abs(expected[k])) for k in expected)
    check(f"{name} reproduces notebook 1 validation metrics", same, {k: round(m[k], 6) for k in expected})
BEST_BASELINE = min(de.BASELINES, key=lambda n: validation_results[n]["overall"]["wape"])
print("best baseline on validation:", BEST_BASELINE)

PASS seasonal_naive_7 reproduces notebook 1 validation metrics | {'wape': 0.412392, 'wpe': -0.115869, 'mae': 0.49337, 'rmse': 0.828341}
PASS moving_average_7 reproduces notebook 1 validation metrics | {'wape': 0.365639, 'wpe': -0.115869, 'mae': 0.437436, 'rmse': 0.738743}
PASS same_dow_mean_4w reproduces notebook 1 validation metrics | {'wape': 0.390235, 'wpe': -0.123295, 'mae': 0.466862, 'rmse': 0.798625}
best baseline on validation: moving_average_7


## 6. Validation experiments

All LightGBM runs train on target days 2024-05-01 to 2024-06-18 and use the validation week for early stopping
(metric: absolute error, which ranks models the same way as WAPE on a fixed set). Because the validation week is used
both for early stopping and for choosing between runs, validation numbers are slightly optimistic; the eval week in
section 8 is the unbiased number.

Runs:
- `l2`, `l1`, `tweedie`: objective comparison with the full feature set (weather excluded).
- `<best>_no_same_day_discount`: best objective without `discount` and `discount_vs_recent`.
- `<best>_oracle_weather`: best objective plus actual same-day weather. Labelled oracle: not available at forecast time.

In [12]:
validation_predictions = {}
validation_boosters = {}


def run_validation(name, objective, weather_mode, same_day_discount):
    cols = dm.model_feature_columns(weather_mode, same_day_discount)
    idx = [col_index[c] for c in cols]
    t0 = time.time()
    booster = dm.train(
        X_all[np.ix_(rows_fit, idx)], y_all[rows_fit], cols, objective, MAX_BOOST_ROUNDS,
        X_all[np.ix_(rows_val, idx)], y_all[rows_val], early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        num_threads=NUM_THREADS,
    )
    seconds = time.time() - t0
    pred = dm.predict(booster, X_all[np.ix_(rows_val, idx)], columns=cols)
    m = de.point_metrics(y_all[rows_val], pred)
    validation_results[name] = {
        "kind": "lightgbm", "objective": objective, "weather_mode": weather_mode,
        "same_day_discount": same_day_discount, "best_iteration": int(booster.best_iteration),
        "train_seconds": round(seconds, 1), "n_features": len(cols), "overall": m,
    }
    validation_predictions[name] = pred
    validation_boosters[name] = booster
    print(f"{name}: wape={m['wape']:.4f} wpe={m['wpe']:+.4f} best_iteration={booster.best_iteration} {seconds:.0f}s")


for objective in ("l2", "l1", "tweedie"):
    run_validation(objective, objective, "none", True)

BEST_OBJECTIVE = min(("l2", "l1", "tweedie"), key=lambda n: validation_results[n]["overall"]["wape"])
print("best objective on validation:", BEST_OBJECTIVE)
for name in list(validation_boosters):
    if name != BEST_OBJECTIVE:
        del validation_boosters[name]

l2: wape=0.3342 wpe=-0.1157 best_iteration=411 152s


l1: wape=0.3333 wpe=-0.1243 best_iteration=1417 560s


tweedie: wape=0.3361 wpe=-0.1225 best_iteration=154 98s
best objective on validation: l1


In [13]:
NO_DISCOUNT_RUN = f"{BEST_OBJECTIVE}_no_same_day_discount"
ORACLE_RUN = f"{BEST_OBJECTIVE}_oracle_weather"
run_validation(NO_DISCOUNT_RUN, BEST_OBJECTIVE, "none", False)
run_validation(ORACLE_RUN, BEST_OBJECTIVE, "oracle", True)

table = pd.DataFrame(
    [{"run": k, "kind": v["kind"], **{m: v["overall"][m] for m in ("wape", "wpe", "mae", "rmse")},
      "best_iteration": v.get("best_iteration")} for k, v in validation_results.items()]
).set_index("run")
print(table.round(4).to_string())

l1_no_same_day_discount: wape=0.3444 wpe=-0.1211 best_iteration=381 179s


l1_oracle_weather: wape=0.3301 wpe=-0.1171 best_iteration=1093 474s
                             kind    wape     wpe     mae    rmse  best_iteration
run                                                                              
seasonal_naive_7         baseline  0.4124 -0.1159  0.4934  0.8283             NaN
moving_average_7         baseline  0.3656 -0.1159  0.4374  0.7387             NaN
same_dow_mean_4w         baseline  0.3902 -0.1233  0.4669  0.7986             NaN
l2                       lightgbm  0.3342 -0.1157  0.3998  0.6851           411.0
l1                       lightgbm  0.3333 -0.1243  0.3988  0.6800          1417.0
tweedie                  lightgbm  0.3361 -0.1225  0.4021  0.6856           154.0
l1_no_same_day_discount  lightgbm  0.3444 -0.1211  0.4120  0.7009           381.0
l1_oracle_weather        lightgbm  0.3301 -0.1171  0.3949  0.6702          1093.0


## 7. Served configuration

Rule fixed before any eval scoring (constant `SERVE_SAME_DAY_DISCOUNT` in section 1):

- Objective: the lowest validation WAPE among `l2`, `l1`, `tweedie`.
- Same-day discount: **excluded** from the served model. Notebook 1 could not confirm that `discount` is set before the
  day (zero-sale and full-stockout days carry `discount = 1.0` far more often), so a model that depends on it may be
  reading realized information. The with-discount model is retrained and scored alongside as a comparison, and the gap
  between the two is the amount of accuracy that rests on that unverified assumption.
- Weather: excluded. The oracle-weather run is reported on validation only.

In [14]:
SERVED_RUN = BEST_OBJECTIVE if SERVE_SAME_DAY_DISCOUNT else NO_DISCOUNT_RUN
COMPARISON_RUN = NO_DISCOUNT_RUN if SERVE_SAME_DAY_DISCOUNT else BEST_OBJECTIVE
served_cfg = validation_results[SERVED_RUN]
comparison_cfg = validation_results[COMPARISON_RUN]
print("served:", SERVED_RUN, "| comparison:", COMPARISON_RUN)

val_frame_served = frame_for(rows_val, validation_predictions[SERVED_RUN])
val_frame_base = frame_for(rows_val, val_baselines[BEST_BASELINE])
validation_results[SERVED_RUN]["breakdowns"] = breakdowns(val_frame_served)
validation_results[BEST_BASELINE]["breakdowns"] = breakdowns(val_frame_base)
for part in ("by_horizon_day", "by_stockout_bucket", "by_activity_flag"):
    a = pd.DataFrame(validation_results[SERVED_RUN]["breakdowns"][part])
    b = pd.DataFrame(validation_results[BEST_BASELINE]["breakdowns"][part])
    key_col = a.columns[0]
    merged = a[[key_col, "n", "wape", "wpe", "mae"]].merge(
        b[[key_col, "wape", "wpe", "mae"]], on=key_col, suffixes=(f"_{SERVED_RUN}", f"_{BEST_BASELINE}"))
    print(f"\nvalidation {part}")
    print(merged.round(4).to_string(index=False))

served: l1_no_same_day_discount | comparison: l1



validation by_horizon_day
horizon_day     n  wape_l1_no_same_day_discount  wpe_l1_no_same_day_discount  mae_l1_no_same_day_discount  wape_moving_average_7  wpe_moving_average_7  mae_moving_average_7
          1 50000                        0.3273                      -0.1512                       0.3624                 0.3161               -0.0447                0.3499
          2 50000                        0.3468                      -0.1521                       0.3701                 0.3328               -0.0088                0.3552
          3 50000                        0.3542                      -0.1748                       0.4134                 0.3385               -0.0937                0.3951
          4 50000                        0.3353                      -0.1302                       0.4755                 0.3898               -0.2543                0.5529
          5 50000                        0.3373                      -0.0941                       0.4773   

## 8. Final retrain and eval scoring (once)

The served and comparison configurations are retrained on target days 2024-05-01 to 2024-06-25 with the number of
boosting rounds found on validation, then scored on 2024-06-26 to 2024-07-02 together with the three baselines.
No choice in this notebook depends on these numbers.

In [15]:
final_boosters = {}
for name, cfg in ((SERVED_RUN, served_cfg), (COMPARISON_RUN, comparison_cfg)):
    cols = dm.model_feature_columns(cfg["weather_mode"], cfg["same_day_discount"])
    idx = [col_index[c] for c in cols]
    t0 = time.time()
    final_boosters[name] = dm.train(X_all[np.ix_(rows_final, idx)], y_all[rows_final], cols, cfg["objective"],
                                    cfg["best_iteration"], num_threads=NUM_THREADS)
    print(f"final {name}: {cfg['best_iteration']} rounds, {time.time() - t0:.0f}s")

eval_predictions = {f"lightgbm_{SERVED_RUN}": None, f"lightgbm_{COMPARISON_RUN}": None}
for name, booster in final_boosters.items():
    cols = booster.feature_name()
    eval_predictions[f"lightgbm_{name}"] = dm.predict(booster, X_all[np.ix_(rows_eval, [col_index[c] for c in cols])], columns=cols)
eval_predictions.update(baseline_predictions(EVAL_START_IDX - 1))

eval_results = {}
for name, pred in eval_predictions.items():
    eval_results[name] = {"overall": de.point_metrics(y_all[rows_eval], pred)}
eval_table = pd.DataFrame([{"model": k, **v["overall"]} for k, v in eval_results.items()]).set_index("model")
print(eval_table[["wape", "wpe", "mae", "rmse", "n"]].round(4).to_string())
SERVED_KEY = f"lightgbm_{SERVED_RUN}"

final l1_no_same_day_discount: 381 rounds, 154s


final l1: 1417 rounds, 550s


                                    wape     wpe     mae    rmse       n
model                                                                   
lightgbm_l1_no_same_day_discount  0.3317 -0.0341  0.3957  0.6730  350000
lightgbm_l1                       0.3238 -0.0465  0.3863  0.6697  350000
seasonal_naive_7                  0.4183  0.0028  0.4991  0.8358  350000
moving_average_7                  0.3612  0.0028  0.4309  0.7228  350000
same_dow_mean_4w                  0.3962 -0.0911  0.4726  0.8129  350000


In [16]:
eval_frame_served = frame_for(rows_eval, eval_predictions[SERVED_KEY])
eval_frame_base = frame_for(rows_eval, eval_predictions[BEST_BASELINE])
eval_results[SERVED_KEY]["breakdowns"] = breakdowns(eval_frame_served)
eval_results[BEST_BASELINE]["breakdowns"] = breakdowns(eval_frame_base)
for part in ("by_horizon_day", "by_stockout_bucket", "by_activity_flag"):
    a = pd.DataFrame(eval_results[SERVED_KEY]["breakdowns"][part])
    b = pd.DataFrame(eval_results[BEST_BASELINE]["breakdowns"][part])
    key_col = a.columns[0]
    merged = a[[key_col, "n", "wape", "wpe", "mae"]].merge(
        b[[key_col, "wape", "wpe", "mae"]], on=key_col, suffixes=("_served", f"_{BEST_BASELINE}"))
    print(f"\neval {part}")
    print(merged.round(4).to_string(index=False))

cat_table = pd.DataFrame(eval_results[SERVED_KEY]["breakdowns"]["by_first_category_id"]).sort_values("sum_actual", ascending=False)
print("\neval by first_category_id, served model (largest 10 by volume)")
print(cat_table.head(10).drop(columns=["sum_actual"]).round(4).to_string(index=False))
worst_stores = worst_groups(eval_frame_served, "store_id")
worst_products = worst_groups(eval_frame_served, "product_id")
eval_results[SERVED_KEY]["worst_stores_by_abs_error"] = worst_stores.to_dict(orient="records")
eval_results[SERVED_KEY]["worst_products_by_abs_error"] = worst_products.to_dict(orient="records")
print("\nstores with the largest total absolute error (served model)")
print(worst_stores.round(4).to_string(index=False))
print("\nproducts with the largest total absolute error (served model)")
print(worst_products.round(4).to_string(index=False))


eval by_horizon_day
horizon_day     n  wape_served  wpe_served  mae_served  wape_moving_average_7  wpe_moving_average_7  mae_moving_average_7
          1 50000       0.3217     -0.0688      0.3537                 0.3389                0.0881                0.3726
          2 50000       0.3387     -0.0958      0.3754                 0.3381                0.0795                0.3747
          3 50000       0.3255     -0.0262      0.3611                 0.3396                0.0786                0.3767
          4 50000       0.3094     -0.0149      0.4206                 0.3450               -0.1200                0.4691
          5 50000       0.3185     -0.0773      0.4843                 0.3755               -0.2130                0.5708
          6 50000       0.3569      0.0358      0.3758                 0.3936                0.1362                0.4144
          7 50000       0.3624      0.0237      0.3992                 0.3976                0.0859                0.4381

ev

Reading the eval results: the full-day stockout bucket has near-zero recorded actuals, so its WAPE and WPE are not
meaningful and MAE is the comparable number. Recorded sales on stockout days understate true demand, so a model that
matches them is also learning that understatement; this bias is reported, not corrected, in this version.

## 9. Feature importance (served model)

In [17]:
served_booster = final_boosters[SERVED_RUN]
importance = pd.DataFrame({
    "feature": served_booster.feature_name(),
    "gain": served_booster.feature_importance("gain"),
    "splits": served_booster.feature_importance("split"),
}).sort_values("gain", ascending=False)
importance["gain_share"] = importance["gain"] / importance["gain"].sum()
print(importance.head(20).round(4).to_string(index=False))

                    feature         gain  splits  gain_share
         sales_mean_14_off7 6071653.8551     512      0.2977
                 product_id 3148388.7928   24536      0.1544
                   store_id 3013800.1494   42137      0.1478
          sales_mean_7_off7 1967653.2245     640      0.0965
         sales_mean_28_off7 1793923.0931     770      0.0880
     sales_same_dow_mean_4w 1107482.6182     565      0.0543
          third_category_id  749977.5345    6744      0.0368
                day_of_week  613876.5095    2732      0.0301
              activity_flag  357723.1345     797      0.0175
                sales_lag_7  247995.9987     958      0.0122
       discount_mean_7_off7  171549.4917    2568      0.0084
     activity_share_28_off7  168564.6944    2164      0.0083
               holiday_flag  143797.7706     763      0.0071
    stock_hours_mean_7_off7  123882.5779    1076      0.0061
                sales_lag_8  122059.9788     498      0.0060
   stock_hours_mean_28_o

## 10. Serving parity, end to end

The server will hold only the last `REQUIRED_HISTORY_DAYS` of history and receive the known-in-advance columns for
the requested days. This cell rebuilds eval features from exactly that input, with the eval week's realized columns
removed, runs the saved model file, and requires predictions identical to section 8.

In [18]:
SERVED_MODEL_PATH = MODEL_DIR / "lgbm_served.txt"
COMPARISON_MODEL_PATH = MODEL_DIR / "lgbm_comparison_with_same_day_discount.txt" if not SERVE_SAME_DAY_DISCOUNT else MODEL_DIR / "lgbm_comparison_without_same_day_discount.txt"
served_booster.save_model(str(SERVED_MODEL_PATH))
final_boosters[COMPARISON_RUN].save_model(str(COMPARISON_MODEL_PATH))
reloaded = lgb.Booster(model_file=str(SERVED_MODEL_PATH))

history_start = EVAL_START - pd.Timedelta(days=dfe.REQUIRED_HISTORY_DAYS)
dates = pd.to_datetime(combined[dfe.DATE_COL])
serving_input = combined[(dates >= history_start)].copy()
serving_input.loc[pd.to_datetime(serving_input[dfe.DATE_COL]) >= EVAL_START, dfe.REALIZED_COLS] = np.nan
serving_features = dfe.build_features(serving_input, weather_mode="none", rows_from_date=EVAL_START)
parity_pred = dm.predict(reloaded, serving_features)
check("serving parity: reloaded model on 34-day history reproduces eval predictions exactly",
      np.array_equal(parity_pred, eval_predictions[SERVED_KEY]), f"{len(parity_pred):,} predictions")

rng = np.random.default_rng(SEED)
sample_idx = np.sort(rng.choice(len(serving_features), size=PARITY_SAMPLE_ROWS, replace=False))
parity_sample = serving_features.iloc[sample_idx][[dfe.KEY_COLS[0], dfe.KEY_COLS[1], dfe.DATE_COL] +
                                                  [c for c in reloaded.feature_name() if c not in dfe.KEY_COLS]].reset_index(drop=True)
parity_sample["prediction"] = parity_pred[sample_idx]

snapshot_start = grid[-1] - pd.Timedelta(days=dfe.REQUIRED_HISTORY_DAYS - 1)
snapshot_cols = list(dict.fromkeys(dfe.STATIC_COLS + [dfe.DATE_COL] + dfe.panel_value_cols("none")))
serving_history = combined.loc[dates >= snapshot_start, snapshot_cols].sort_values(dfe.KEY_COLS + [dfe.DATE_COL]).reset_index(drop=True)
future_dates = pd.date_range(grid[-1] + pd.Timedelta(days=1), periods=HORIZON_DAYS, freq="D")
future = serving_history.drop_duplicates(dfe.KEY_COLS)[dfe.STATIC_COLS].merge(
    pd.DataFrame({dfe.DATE_COL: future_dates.strftime("%Y-%m-%d")}), how="cross")
for c in dfe.REALIZED_COLS:
    if c in snapshot_cols:
        future[c] = np.nan
future["activity_flag"] = 0
future["holiday_flag"] = (future_dates.dayofweek[np.tile(np.arange(HORIZON_DAYS), n_series)] >= 5).astype(int)
future["discount"] = np.nan
future_features = dfe.build_features(pd.concat([serving_history, future], ignore_index=True), rows_from_date=future_dates[0])
future_pred = dm.predict(reloaded, future_features)
check("snapshot alone produces complete features and finite predictions for the next 7 days",
      not future_features[reloaded.feature_name()].isna().any().any() and np.isfinite(future_pred).all(),
      f"{len(future_pred):,} rows, dates {future_dates[0].date()}..{future_dates[-1].date()}, activity_flag=0, holiday_flag=weekend")

PASS serving parity: reloaded model on 34-day history reproduces eval predictions exactly | 350,000 predictions


PASS snapshot alone produces complete features and finite predictions for the next 7 days | 350,000 rows, dates 2024-07-03..2024-07-09, activity_flag=0, holiday_flag=weekend


## 11. Save artifacts

In [19]:
for module in ("demand_features.py", "demand_eval.py", "demand_model.py"):
    src = MODULE_DIR / module
    if src.resolve() != (OUTPUT_DIR / module).resolve():
        shutil.copy2(src, OUTPUT_DIR / module)

environment["module_sha256"] = {m: sha256_of(MODULE_DIR / m) for m in ("demand_features.py", "demand_eval.py", "demand_model.py")}
model_card = {
    "dataset": {"repo": DATASET_REPO, "revision": DATASET_REVISION, "license": "CC BY 4.0"},
    "served_run": SERVED_RUN,
    "comparison_run": COMPARISON_RUN,
    "objective": served_cfg["objective"],
    "params": dm.model_params(served_cfg["objective"]) | {"num_threads": NUM_THREADS},
    "num_boost_round": served_cfg["best_iteration"],
    "feature_columns": served_booster.feature_name(),
    "categorical_features": [c for c in dm.CATEGORICAL_FEATURES if c in served_booster.feature_name()],
    "same_day_discount": served_cfg["same_day_discount"],
    "weather_mode": served_cfg["weather_mode"],
    "horizon_days": dfe.HORIZON,
    "min_lag_days": dfe.MIN_LAG,
    "required_history_days": dfe.REQUIRED_HISTORY_DAYS,
    "train_target_days": split["final_train_target_days"],
    "prediction_clip_min": 0.0,
    "model_files": {
        SERVED_MODEL_PATH.name: sha256_of(SERVED_MODEL_PATH),
        COMPARISON_MODEL_PATH.name: sha256_of(COMPARISON_MODEL_PATH),
    },
    "eval_metrics_served": eval_results[SERVED_KEY]["overall"],
    "known_limitations": [
        "Actuals on stockout days are censored; recorded-sales bias is reported, not corrected.",
        "Features use history up to t-7 only; the most recent 6 days are unused for near horizons.",
        "Sales values are normalized by the dataset publisher, so errors are not in physical units.",
        "Whether same-day discount is known in advance is unverified; the served model excludes it."
        if not SERVE_SAME_DAY_DISCOUNT else "Served model assumes same-day discount is known in advance (unverified).",
    ],
}
metrics = {"split": split, "validation": validation_results, "eval": eval_results,
           "served_run": SERVED_RUN, "comparison_run": COMPARISON_RUN, "best_baseline_on_validation": BEST_BASELINE}

(OUTPUT_DIR / "nb2_model_card.json").write_text(json.dumps(model_card, indent=2, default=str))
(OUTPUT_DIR / "nb2_metrics.json").write_text(json.dumps(metrics, indent=2, default=str))
(OUTPUT_DIR / "nb2_environment.json").write_text(json.dumps(environment, indent=2))
importance.to_csv(OUTPUT_DIR / "nb2_feature_importance.csv", index=False)
parity_sample.to_parquet(OUTPUT_DIR / "nb2_parity_sample.parquet", index=False)
serving_history.to_parquet(OUTPUT_DIR / "nb2_serving_history.parquet", index=False)
eval_out = pd.DataFrame({"store_id": meta["store_id"].to_numpy()[rows_eval], "product_id": meta["product_id"].to_numpy()[rows_eval],
                         "dt": meta["dt"].to_numpy()[rows_eval], "actual": y_all[rows_eval],
                         "stockout_hours": stock_all[rows_eval]})
for name, pred in eval_predictions.items():
    eval_out[f"pred_{name}"] = pred
eval_out.to_parquet(OUTPUT_DIR / "nb2_eval_predictions.parquet", index=False)
(OUTPUT_DIR / "nb2_checks.json").write_text(json.dumps(report, indent=2, default=str))

for p in sorted(list(OUTPUT_DIR.glob("nb2_*")) + list(MODEL_DIR.glob("*.txt"))):
    print(f"{p.relative_to(OUTPUT_DIR)}: {p.stat().st_size:,} bytes")
failed = [k for k, v in report["checks"].items() if not v["passed"]]
print(f"checks: {len(report['checks'])} run, {len(failed)} failed")

models/lgbm_comparison_with_same_day_discount.txt: 70,557,745 bytes
models/lgbm_served.txt: 18,736,412 bytes
nb2_checks.json: 3,198 bytes
nb2_environment.json: 494 bytes
nb2_eval_predictions.parquet: 8,256,355 bytes
nb2_feature_importance.csv: 2,028 bytes
nb2_metrics.json: 60,879 bytes
nb2_model_card.json: 2,751 bytes
nb2_parity_sample.parquet: 133,431 bytes
nb2_serving_history.parquet: 5,256,853 bytes
checks: 17 run, 0 failed
